# ✅ Solutions — Chapter 3 — Description Logics — agentic lab

This is the **solution** notebook: the same lab with all 3 tasks worked. It runs clean end to end, which is what proves the reference implementations satisfy the marking scheme.

> Student version: [`05_agentic_lab.ipynb`](05_agentic_lab.ipynb)

# Chapter 3 — Description Logics
### Notebook 5 · Agentic lab — naming the logic, and paying for soundness

*Book reference: Extends §3.2–3.3*

Two firsts for this course. The agent's oracle is **free and always right**, so failures label themselves — the cleanest self-improvement setting we have. And the MDP is **stochastic**, because the choice is between a cheap guess and an expensive certainty.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch03_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
from oe_course.skills import Skill
from oe_course.selfimprove import SelfImprovingSkill
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Expose a reasoner as a **function tool** and make an agent use it instead of guessing.
2. Build a dataset whose verdicts are **balanced**, and see why an unbalanced one silently teaches nothing.
3. Model the sound-but-costly / cheap-but-unsound choice as a **stochastic MDP** and solve it exactly.
4. Run a self-improvement loop where the oracle supplies the labels.

> **Prerequisite:** the Chapter 1 agentic lab. The discipline is the same; the task changes.

## 1. Tools: the reasoner is one of them

The interesting tool here is `check_subsumption`. Its description tells the agent *when* to reach for it — "instead of reasoning by eye" — because the failure mode this lab is built around is an agent that answers from the shape of the axioms rather than from a proof.

In [ ]:
ctx = AG.Ch3Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:22s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":22s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['load_kb'].invoke({'name': 'wildlife'}))
print(tools['dl_expressivity'].invoke({}))
print(tools['check_subsumption'].invoke({'sub': 'Giraffe', 'sup': 'Animal'}))
print(tools['check_satisfiability'].invoke({'concept_a': 'Giraffe',
                                            'concept_b': 'Carnivore'}))
print('\ntrajectory:', ctx.log.names())

## 2. The dataset, and why its balance matters

Ten knowledge bases. Each asks for a DL name **and** a subsumption verdict.

The verdicts are deliberately **half true and half false**. An all-true set — which is what you get if you write the queries carelessly — would let the strategy *"assume it follows"* score full marks on that half. The rule about actually running the reasoner would never be punished, and the optimiser would never learn it. **A dataset that cannot punish a mistake cannot teach it.**

In [ ]:
all_examples = AG.build_dataset('all')
print(pd.DataFrame([{'id': e.id, 'DL': e.gold_dl, 'query': e.query,
                     'holds': e.gold_subsumption} for e in all_examples]
                   ).to_string(index=False))
verdicts = [e.gold_subsumption for e in all_examples]
print(f'\nbalance: {sum(verdicts)} hold, {len(verdicts) - sum(verdicts)} do not')

In [ ]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print('train:', [e.id for e in train])
print('dev  :', [e.id for e in dev])
print('\nThe split is stratified (alternating), not sequential: a first-six/\n'
      'last-four cut would put every transitive and inverse case in train and\n'
      'leave dev nearly trivial, so the held-out score would flatter the agent.')

In [ ]:
print('KB as the agent sees it (no logic named for it):\n')
print(dev[1].kb)
print('\nquery:', dev[1].query)

## 3. Baseline and GEPA

The metric gives half a mark for the DL name and half for the verdict, and names the specific letter that was missed — `s-for-transitive`, `i-for-inverse` and so on — so the optimiser learns *which* rule of §3.2 it broke.

In [ ]:
lm = llm.configure_dspy(AG.DL_RULEBOOK, AG.dl_responder)
baseline = AG.DLProgram()
example = train[-1]
pred = baseline(**example.inputs())
print('KB id      :', example.id)
print('answered   :', pred.dl, '/', pred.subsumption)
print('correct    :', example.gold_dl, '/', example.gold_subsumption)
report = AG.dl_scorer(example, pred)
print('score      :', report.score)
for n in report.notes:
    print('   ', n)
print('violated   :', report.violated)

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.dl_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.dl_scorer, AG.DL_RULEBOOK)
reflect = llm.reflection_lm(AG.DL_RULEBOOK, AG.dl_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(AG.DLProgram(), tuned, dev, AG.dl_scorer)
print(result.report())

In [ ]:
found = AG.DL_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(AG.DL_RULEBOOK.ids) - found))

> Note what `use-the-reasoner` being discovered actually means: the optimiser learned to make the agent **call a tool** rather than answer from the prompt. That is prompt optimisation changing *behaviour*, not just wording — and it only happened because half the verdicts were false.

## 4. A stochastic MDP: when is the reasoner worth calling?

Every MDP so far has had deterministic transitions. This one does not.

The agent answers a series of subsumption queries. For each it may:

* **guess** — free, correct with probability `pᵢ` (a structural heuristic such as "it is asserted in the hierarchy, so it must follow");
* **reason** — always correct, costs `reasoner_cost`, and consumes one unit of a limited budget.

| | |
|---|---|
| **S** | which query we are on, and how much budget is spent |
| **A** | `guess` or `reason` |
| **T** | **stochastic** — guessing lands in the same next state, but the reward is Bernoulli |
| **R** | 1 for a correct answer; `1 − cost` for reasoning |

This is §3.3's complexity discussion as a decision problem: soundness is a purchase, and the question is where to spend.

In [ ]:
accuracy = [0.9, 0.5, 0.95, 0.6]     # heuristic reliability, per query
M = AG.ReasoningBudgetMDP(accuracy, reasoner_cost=0.2, budget=2)
V, pi = mdp.value_iteration(M)
print(f'V*(s0) = {V[M.initial_state()]:.3f}')
print('optimal plan:', M.optimal_plan(pi))
print('heuristic accuracies:', accuracy)
print(f'\nalways guess : {sum(accuracy):.3f}')
print(f'always reason: {len(accuracy) * (1 - 0.2):.3f}  (if budget allowed)')

The optimal policy spends its two reasoner calls on queries **2 and 4** — the ones where the heuristic is least reliable (0.5 and 0.6) — and guesses on the two where it is nearly always right (0.9, 0.95). Nobody told it that rule; value iteration derived it from the reward.

The threshold is worth stating exactly: **call the reasoner when `1 − cost > pᵢ`**, i.e. when the heuristic's error rate exceeds the reasoner's price.

In [ ]:
rows = []
for cost in [0.0, 0.1, 0.3, 0.5, 0.7]:
    Mc = AG.ReasoningBudgetMDP(accuracy, reasoner_cost=cost, budget=4)
    Vc, pic = mdp.value_iteration(Mc)
    plan = Mc.optimal_plan(pic)
    rows.append({'reasoner_cost': cost, 'V*': round(Vc[Mc.initial_state()], 3),
                 'reasoner calls': plan.count('reason'), 'plan': ' '.join(plan)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nWith an unlimited budget the agent still declines to reason when the\n'
      'price exceeds the heuristic error rate. "Always be sound" is not the\n'
      'optimal policy under a cost model -- which is uncomfortable, and true.')

## 5. Self-improvement, with a free oracle

This is the chapter where self-improvement is genuinely easy: the tableau settles every question, so **every failure labels itself**. No human in the loop, no annotation budget.

The promotion gate still matters. An oracle removes the labelling problem; it does not remove the risk of adopting a candidate that happens to score well on the data it was tuned on.

In [ ]:
skill = Skill(
    name='dl-analyst',
    description='Name the DL a knowledge base needs and answer subsumption soundly.',
    build=lambda instruction: AG.DLProgram(instruction),
    scorer=AG.dl_scorer,
    dataset=dev,
    instruction=AG.BASELINE_INSTRUCTION,
    tools=['load_kb', 'dl_expressivity', 'check_subsumption', 'tableau_trace'],
)
skill.evaluate()
print(skill.card())

In [ ]:
sis = SelfImprovingSkill(
    skill, holdout=dev,
    optimise=lambda prog, tr: opt.run_gepa(prog, tr, gepa_metric,
                                          max_metric_calls=60, reflection_lm=reflect),
    min_gain=0.01)
sis.run_all(train)
print(sis.report())

In [ ]:
print(sis.improve())
print(sis.improve())
print()
print(skill.card())

> The second round is **rejected** — no gain on the holdout, so the skill stays where it is. A self-improvement loop that never rejects a candidate is not improving; it is drifting.

### Task 5.1 — Break the dataset on purpose

Rebuild the dataset with **all verdicts true**, re-run GEPA, and report whether `use-the-reasoner` is still discovered. Explain the result.

> **Hint.** Filter `train` down to the examples whose gold verdict is True.

In [ ]:
import copy
biased_train = []
for e in train:
    if e.gold_subsumption:
        biased_train.append(e)
print(f'biased train has {len(biased_train)} examples, all with verdict True')

tuned_biased = opt.run_gepa(AG.DLProgram(), biased_train, gepa_metric,
                            valset=biased_train, max_metric_calls=60,
                            reflection_lm=reflect)
found_biased = AG.DL_RULEBOOK.active_in(opt.instruction_of(tuned_biased))
print('rules discovered:', sorted(found_biased))
print('use-the-reasoner found?', 'use-the-reasoner' in found_biased)
print('\nWith every verdict true, guessing "true" is never wrong, so the metric\n'
      'never complains and the rule is never learned. The agent would then fail\n'
      'silently in production on the first negative case. Dataset balance is not\n'
      'hygiene -- it decides what your agent is capable of learning.')

**Checks for Task 5.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert 'use-the-reasoner' not in found_biased

### Task 5.2 — Find the price at which soundness stops paying

For a single query with heuristic accuracy `p`, derive the reasoner cost at which guessing and reasoning are equally good, and confirm it numerically.

> **Hint.** Guessing is worth p; reasoning is worth 1 − cost.

In [ ]:
rows = []
for p in [0.5, 0.7, 0.9]:
    for cost in [0.05, 0.1, 0.2, 0.3, 0.5]:
        Mx = AG.ReasoningBudgetMDP([p], reasoner_cost=cost, budget=1)
        Vx, pix = mdp.value_iteration(Mx)
        rows.append({'p': p, 'cost': cost,
                     'action': Mx.optimal_plan(pix)[0],
                     'V*': round(Vx[Mx.initial_state()], 3),
                     'predicted': 'reason' if (1 - cost) > p else 'guess'})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print('\nReasoning is worth it exactly when 1 - cost > p, i.e. cost < 1 - p:\n'
      'the reasoner is worth its price precisely when the heuristic error rate\n'
      'exceeds it. Value iteration and the algebra agree on every row.')

**Checks for Task 5.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert (df['action'] == df['predicted']).all()

### Task 5.3 — Add a rule the agent must learn from the reasoner

Add a `check-unsatisfiable-classes` rule: before answering, the agent should verify the queried concepts are satisfiable at all. Build a KB where ignoring this gives a misleading answer, and show the metric punishing it.

In [ ]:
# In the wildlife KB, Giraffe and Carnivore is unsatisfiable -- so it is
# vacuously subsumed by EVERYTHING, including nonsense.
w = dl.wildlife_tbox()
impossible = dl.And(A('Giraffe'), A('Carnivore'))
print('Giraffe-and-Carnivore satisfiable?',
      dl.satisfiable(impossible, w).satisfiable)
print('...subsumed by Plant?    ', dl.subsumes(impossible, A('Plant'), w))
print('...subsumed by Bottom?   ', dl.subsumes(impossible, dl.Bottom, w))
print('\nAn unsatisfiable concept is subsumed by everything -- ex falso quodlibet.\n'
      'So a "yes" verdict here is TRUE and completely useless: the honest answer\n'
      'is "the question is malformed, that class can have no instances". An agent\n'
      'that reports the entailment without the satisfiability check gives a\n'
      'technically correct answer that will mislead its user -- which is a good\n'
      'argument for making the check part of the skill rather than the prompt.')

**Checks for Task 5.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert dl.subsumes(impossible, A('Plant'), w)

## Chapter 3 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 3 | Ch. 4 |
|---|---|---|---|---|
| task | assess | formalise | name + entail | build an axiom |
| MDP | gather evidence | search a proof | **budgeted, stochastic** | construct |
| grader | labels + judge | decision procedure | **free oracle** | labels + profiles |
| GEPA learns | reporting | quantifier semantics | DL letters + tool use | profile limits |

Chapter 3's contribution to the course argument: when a sound oracle exists, use it — for labels, for self-improvement, and as the thing your agent is *taught to call*. Chapter 4 then asks what happens when you standardise one of these logics into a language committees must agree on.